# Notebook 7: Retrieval Augmented Generation (RAG) – Smarter Answers from Your Data

**🎯 Goal:** Combine a vectorstore with an LLM to build a complete RAG pipeline that can answer questions based on the content of your private documents.

## 🧩 Concept Overview

**Retrieval Augmented Generation (RAG)** is the process of giving an LLM access to external knowledge. Instead of relying only on its training data, the LLM can answer questions about specific, up-to-date, or private information.

It works in two simple steps:
1.  **Retrieve:** When you ask a question, the system first searches your vectorstore to find the most relevant document chunks. This is the "Retrieval" step.
2.  **Augment & Generate:** The system then takes your question and the retrieved document chunks and puts them both into a prompt for the LLM. The LLM uses this new, "augmented" information to generate a final answer.

This prevents hallucination and allows the LLM to work with data it has never seen before.

## 🖼️ Visual Diagram

The complete RAG workflow:

```
                                   +-------------------------+
                                   | Your Documents          |
                                   | (e.g., PDFs, TXTs)      |
                                   +-----------+-------------+
                                               | (Embed & Store)
                                               v
                                   +-----------+-------------+
Your Question --- (1. Embed) ---> | Vectorstore             | -- (2. Retrieve) --> Relevant Chunks
                                   +-------------------------+                           |
                                                                                         |
                                    (3. Augment Prompt)                                  |
                                               +-----------------------------------------+
                                               |
                                               v
                 +--------------------------------------------------------------------+
                 | Prompt: "Based on this context... [Relevant Chunks]... answer the  |
                 | question: [Your Question]"                                        |
                 +---------------------------+----------------------------------------+
                                             |
                                             v
                                   +---------+---------+
                                   |       LLM         | -- (4. Generate) --> Final Answer
                                   +-------------------+
```

## ⚙️ Setup

Let's set up our environment and create the same vectorstore from the previous notebook.

In [ ]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain.chains import RetrievalQA

load_dotenv()

# Initialize LLM and Embeddings
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.3)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# Load and split the document
loader = TextLoader('../data/annual-report.pdf')
documents = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
chunks = text_splitter.split_documents(documents)

# Create the vectorstore
vectorstore = FAISS.from_documents(chunks, embeddings)

print("Vectorstore created and ready.")

## 1. The Retriever

A `retriever` is a component that, given a query, returns relevant documents. Vectorstores are the most common type of retriever. We can easily convert our vectorstore into a retriever interface.

In [ ]:
# 🗣️ Convert the vectorstore into a retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 3}) # Get top 3 results

# Let's test the retriever directly
query = "What is Project Chimera?"
retrieved_docs = retriever.invoke(query)

print(f"Retrieved {len(retrieved_docs)} documents.")
print("\n--- First Retrieved Document ---")
print(retrieved_docs[0].page_content)

## 2. The `RetrievalQA` Chain

This is the workhorse chain for RAG. It takes a retriever and an LLM and combines them to answer questions.

The default `chain_type="stuff"` does exactly what we described: it "stuffs" all the retrieved documents into the prompt. This is simple and effective for a small number of documents.

In [ ]:
# ⛓️ Create the RetrievalQA chain
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff", # Other options: "map_reduce", "refine", "map_rerank"
    retriever=retriever,
    return_source_documents=True # We want to see which chunks were used
)

# 🚀 Run the chain
response = qa_chain.invoke({"query": "What were the total revenues and net profits in 2024?"})

print("--- Final Answer ---")
print(response["result"])

print("\n--- Source Documents Used ---")
for doc in response["source_documents"]:
    print(f"- {doc.page_content[:200]}...")

## 🔧 Hands-On Exercise

**Goal:** Build a RAG chain to ask questions about the movie plots dataset.

1.  Load `../data/movie-plots.csv` with `CSVLoader` (use `source_column='Title'`).
2.  Split it into chunks (size 400, overlap 50).
3.  Create a `FAISS` vectorstore using `HuggingFaceEmbeddings`.
4.  Create a `RetrievalQA` chain.
5.  Ask it: `"What is the plot of the movie 'Star Voyager'?'"` and print the result.

In [ ]:
from langchain_community.document_loaders import CSVLoader
from langchain_community.embeddings import HuggingFaceEmbeddings

# 1. Load
movie_loader = CSVLoader('../data/movie-plots.csv', source_column='Title')
movie_docs = movie_loader.load()

# 2. Split
movie_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)
movie_chunks = movie_splitter.split_documents(movie_docs)

# 3. Embed and Store
hf_embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
movie_db = FAISS.from_documents(movie_chunks, hf_embeddings)

# 4. Create RAG chain
movie_qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=movie_db.as_retriever(search_kwargs={"k": 2})
)

# 5. Ask a question
movie_query = "What is the plot of the movie 'Star Voyager'?"
movie_response = movie_qa_chain.invoke({"query": movie_query})

print(movie_response["result"])

## 🐞 Bug Bounty

A common issue is not retrieving enough documents (`k` is too low) to answer a question that requires combining information from multiple chunks.

**Task:** Ask a question about both 'Project Chimera' and 'Project Terra'. First, try it with a retriever that only fetches `k=1` document. It will likely fail or only answer one part of the question. Then, fix it by setting `k=2` or `k=3`.

In [ ]:
# --- BROKEN CODE (k is too low) ---
buggy_retriever = vectorstore.as_retriever(search_kwargs={"k": 1})
buggy_qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=buggy_retriever
)

complex_query = "Tell me about Project Chimera and Project Terra."
buggy_response = buggy_qa_chain.invoke({"query": complex_query})

print("--- Buggy Response (k=1) ---")
print(buggy_response["result"])

# --- FIXED CODE ---
fixed_retriever = vectorstore.as_retriever(search_kwargs={"k": 2})
fixed_qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=fixed_retriever
)
fixed_response = fixed_qa_chain.invoke({"query": complex_query})
print("\n--- Fixed Response (k=2) ---")
print(fixed_response["result"])

## 💡 Pro Tip

The `stuff` chain type is simple, but it can fail if you retrieve too many documents and exceed the LLM's context window. For those cases, you can use `chain_type="map_reduce"`. It works by:
1.  **Map:** Sending each retrieved document to the LLM individually with the question.
2.  **Reduce:** Taking all the individual answers and sending them to the LLM in a final call to synthesize them into a single answer.

It's slower and more expensive (more LLM calls), but it can scale to any number of documents.

## 🏁 Summary

You've built a complete RAG application, the most common and powerful pattern for building LLM apps with custom data!

1.  **RAG = Retrieve + Augment + Generate:** It's a simple but powerful two-step process to make LLMs smarter with your data.
2.  **`RetrievalQA` is your friend:** This chain orchestrates the entire RAG workflow for you.
3.  **Always check your sources:** Using `return_source_documents=True` is critical for building trust and allowing users to verify the AI's answers.